# Unit 6: Supply Chains - Nurse Scheduling QUBO Micro-Example

**Companion notebook for *What Quantum Computers Are Actually For***

**Notebook class:** toy demonstration


**Code note:** This notebook is written as teaching code rather than library code. The Quokka calls, circuit construction, and measurement post-processing stay explicit on purpose so you can inspect the mechanism end to end. A production library would factor more of this behind helpers; here we keep the moving parts visible.

This notebook narrows the scheduling story to a 2-nurse, 2-shift micro-example so the QUBO penalties and the QUBO-to-Ising map stay transparent.

It does **not** model the 8-nurse chapter example or a realistic hospital roster. It shows the smallest scheduling instance where a hard constraint and a soft preference can be written exactly and then fed into a one-layer QAOA circuit.

**What you'll see:**
1. A 2-nurse, 2-shift scheduling micro-example
2. The exact QUBO cost table and its optimal schedules
3. An exact QUBO-to-Ising conversion for the same micro-example
4. A one-layer QAOA parameter search and a Quokka run that biases samples toward feasible schedules

In [ ]:
import json

import itertools

import numpy as np

import matplotlib.pyplot as plt



import requests

from requests.packages.urllib3.exceptions import InsecureRequestWarning

requests.packages.urllib3.disable_warnings(InsecureRequestWarning)



QUOKKA = "quokka1"

QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"



def _decode_quokka_counts(payload: dict) -> dict:
    """Normalize Quokka responses to a simple {bitstring: count} mapping."""
    if isinstance(payload, dict) and all(isinstance(v, int) for v in payload.values()):
        return payload

    if not isinstance(payload, dict):
        raise TypeError(f"Unexpected Quokka payload type: {type(payload)!r}")

    if payload.get("error_code", 0) not in (0, None):
        raise RuntimeError(f"Quokka error {payload.get('error_code')}: {payload.get('error')}")

    result = payload.get("result")
    if not isinstance(result, dict) or "c" not in result:
        raise ValueError(f"Unexpected Quokka result format: {payload}")

    counts = {}
    for shot in result["c"]:
        bitstring = ''.join(str(int(bit)) for bit in shot)
        counts[bitstring] = counts.get(bitstring, 0) + 1
    return counts

def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(
        QUOKKA_URL,
        json={"script": program, "count": shots},
        verify=False,
        timeout=30,
    )
    result.raise_for_status()
    return _decode_quokka_counts(result.json())

print(f"Connected to {QUOKKA_URL}")

## 1. Define the scheduling micro-example

Nurse A and Nurse B must cover two shifts: Mon-day and Mon-night.

Hard constraint: Nurse A must take exactly one of the two shifts, so Nurse B automatically takes the other one.

Soft constraint: Nurse A prefers the day shift, so assigning Nurse A to the night shift carries a small penalty.

Binary variables:
- x0 = 1 if Nurse A takes Mon-day
- x1 = 1 if Nurse A takes Mon-night

In [ ]:
nurses = ['Nurse A', 'Nurse B']
shifts = ['Mon-day', 'Mon-night']
P = 5.0
pref = 1.0


def qubo_cost(bits):
    x0, x1 = bits
    return P * (x0 + x1 - 1) ** 2 + pref * x1


def describe_assignment(bits):
    x0, x1 = bits
    if x0 + x1 != 1:
        return 'infeasible'
    if x0 == 1:
        return 'Nurse A -> Mon-day, Nurse B -> Mon-night'
    return 'Nurse B -> Mon-day, Nurse A -> Mon-night'


# Symmetric QUBO matrix for x^T Q x.
Q = np.array([
    [-P, P],
    [P, -P + pref],
])

print('QUBO matrix Q:')
print(Q)
print()
print('Exact micro-example costs:')
print(f"{'x0':>3} {'x1':>3} | {'cost':>6} | {'status':>10} | schedule")
print('-' * 72)

classical_records = []
for x0, x1 in itertools.product([0, 1], repeat=2):
    bits = (x0, x1)
    cost = qubo_cost(bits)
    feasible = (x0 + x1 == 1)
    classical_records.append((bits, cost, feasible))
    status = 'feasible' if feasible else 'invalid'
    print(f"{x0:>3} {x1:>3} | {cost:>6.1f} | {status:>10} | {describe_assignment(bits)}")

best_classical_bits, best_classical_cost, _ = min(classical_records, key=lambda item: item[1])
feasible_states = {'01', '10'}
print()
print(f"Best classical assignment: {best_classical_bits} with cost {best_classical_cost:.1f}")

## 2. Exact Ising map and one-layer QAOA

For this two-variable micro-example, we can derive the Ising Hamiltonian exactly by matching the QUBO energy on all four computational basis states.

We then classically tune one layer of QAOA on that exact Ising model and run the tuned circuit on Quokka. The honest claim is modest: the circuit should bias sampling toward the two feasible schedules, not solve realistic nurse rostering.

In [ ]:
from scipy.linalg import expm

# Solve the exact Ising coefficients from the four QUBO energies.
rows = []
energies = []
for x0, x1 in itertools.product([0, 1], repeat=2):
    z0 = 1 - 2 * x0
    z1 = 1 - 2 * x1
    rows.append([1.0, z0, z1, z0 * z1])
    energies.append(qubo_cost((x0, x1)))

const, h0, h1, J01 = np.linalg.solve(np.array(rows), np.array(energies))
print(f"Ising coefficients: const={const:.2f}, h0={h0:.2f}, h1={h1:.2f}, J01={J01:.2f}")

I = np.eye(2)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
problem_hamiltonian = (
    const * np.kron(I, I)
    + h0 * np.kron(Z, I)
    + h1 * np.kron(I, Z)
    + J01 * np.kron(Z, Z)
)
mixer_hamiltonian = np.kron(X, I) + np.kron(I, X)
plus_state = np.ones(4, dtype=complex) / 2

best = None
for gamma in np.linspace(0, np.pi, 121):
    U_problem = expm(-1j * gamma * problem_hamiltonian)
    for beta in np.linspace(0, np.pi / 2, 121):
        U_mixer = expm(-1j * beta * mixer_hamiltonian)
        state = U_mixer @ (U_problem @ plus_state)
        probabilities = np.abs(state) ** 2
        feasible_probability_exact = float(probabilities[1] + probabilities[2])
        expected_cost_exact = float(sum(
            probabilities[index] * qubo_cost(((index >> 1) & 1, index & 1))
            for index in range(4)
        ))
        score = (-feasible_probability_exact, expected_cost_exact)
        if best is None or score < best[0]:
            best = (score, gamma, beta, feasible_probability_exact, expected_cost_exact)

_, gamma, beta, feasible_probability_exact, expected_cost_exact = best
print(f"Best one-layer parameters from exact search: gamma={gamma:.4f}, beta={beta:.4f}")
print(f"Exact feasible-state probability with those parameters: {feasible_probability_exact:.3f}")
print(f"Exact expected QUBO cost with those parameters:        {expected_cost_exact:.3f}")

qaoa = f"""OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];

h q[0];
h q[1];

cx q[0], q[1];
rz({2 * gamma * J01:.6f}) q[1];
cx q[0], q[1];

rz({2 * gamma * h0:.6f}) q[0];
rz({2 * gamma * h1:.6f}) q[1];

rx({2 * beta:.6f}) q[0];
rx({2 * beta:.6f}) q[1];

measure q[0] -> c[0];
measure q[1] -> c[1];
"""

results = run_qasm(qaoa, shots=1024)
feasible_probability = sum(count for state, count in results.items() if state in feasible_states) / sum(results.values())
best_outcome = max(results, key=results.get)
qaoa_expected_cost = sum(results[state] * qubo_cost((int(state[0]), int(state[1]))) for state in results) / sum(results.values())

print()
print('Quokka measurement results:')
for outcome in sorted(results.keys(), key=lambda state: results[state], reverse=True):
    status = 'feasible' if outcome in feasible_states else 'invalid'
    schedule = describe_assignment((int(outcome[0]), int(outcome[1])))
    print(f"  {outcome}: {results[outcome]:>4} counts | {status:>8} | cost {qubo_cost((int(outcome[0]), int(outcome[1]))):.1f} | {schedule}")

print()
print(f"Dominant measured outcome:          {best_outcome}")
print(f"Measured feasible-state probability: {feasible_probability:.3f}")
print(f"Measured expected QUBO cost:         {qaoa_expected_cost:.3f}")

labels = sorted(results.keys())
counts = [results[label] for label in labels]
colors = ['#009688' if label in feasible_states else '#B0BEC5' for label in labels]
plt.figure(figsize=(8, 4))
plt.bar(labels, counts, color=colors)
plt.xlabel('Measured bitstring')
plt.ylabel('Counts')
plt.title('One-layer QAOA on the scheduling micro-example')
plt.legend(handles=[
    plt.Rectangle((0, 0), 1, 1, color='#009688', label='Feasible schedules'),
    plt.Rectangle((0, 0), 1, 1, color='#B0BEC5', label='Invalid schedules'),
])
plt.tight_layout()
plt.show()

## What's next?

- [Deep-Dive 6 - QUBO Engineering](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/12-qubo-engineering.md): the penalty-design details behind this micro-example
- Scale the same construction to more nurses, more shifts, and more constraint types
- Compare this gate-based toy with classical simulated annealing and quantum annealing
- [Unit 7 - Materials Science](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/13-materials-science.md): where the target Hamiltonian comes from physics rather than scheduling penalties